# Cityscapes semantic segmentation — train on Colab

Runs the full pipeline from this repository on a Colab GPU: download Cityscapes, train, evaluate, benchmark and export predictions.

Checkpoints and logs are written to Google Drive, so a disconnected session can be resumed by re-running the notebook.

**Before starting:** `Runtime → Change runtime type → GPU`.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

## 1. Setup

In [ ]:
!git clone https://github.com/vertikasingh123/autonomous_driving.git /content/autonomous_driving
%cd /content/autonomous_driving
!pip install -q -e ".[data]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/data/cityscapes'
OUTPUT_DIR = '/content/drive/MyDrive/cityseg/unet_r34'   # survives disconnects
CONFIG = 'configs/unet_r34.yaml'
CHECKPOINT = f'{OUTPUT_DIR}/checkpoints/best.ckpt'

## 2. Download Cityscapes

Needs a free account at https://www.cityscapes-dataset.com/ — you are prompted for the username and password. About 11 GB; takes a few minutes.

In [ ]:
import os
from cityscapesscripts.download import downloader

if not os.path.isdir(f'{DATA_ROOT}/leftImg8bit/train'):
    os.makedirs(DATA_ROOT, exist_ok=True)
    session = downloader.login()
    downloader.download_packages(
        session=session,
        package_names=['gtFine_trainvaltest.zip', 'leftImg8bit_trainvaltest.zip'],
        destination_path=DATA_ROOT,
    )
    !unzip -q -o {DATA_ROOT}/gtFine_trainvaltest.zip -d {DATA_ROOT}
    !unzip -q -o {DATA_ROOT}/leftImg8bit_trainvaltest.zip -d {DATA_ROOT}
    !rm {DATA_ROOT}/*.zip
!ls {DATA_ROOT}

## 3. Train

`--resume auto` continues from the last checkpoint on Drive if one exists, so this cell can simply be re-run after a disconnect.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tensorboard

In [ ]:
!python -m cityseg.train --config {CONFIG} --data-root {DATA_ROOT} --output-dir {OUTPUT_DIR} --num-workers 2 --resume auto

## 4. Evaluate

Full-resolution (1024×2048) evaluation on the 500 validation images. Writes `metrics.json` and `results.md` next to the checkpoint — paste `results.md` into the README.

In [ ]:
!python -m cityseg.evaluate --checkpoint {CHECKPOINT} --data-root {DATA_ROOT} --num-workers 2

In [ ]:
# With horizontal-flip test-time augmentation
!python -m cityseg.evaluate --checkpoint {CHECKPOINT} --data-root {DATA_ROOT} --num-workers 2 --flip --output-dir {OUTPUT_DIR}/eval_flip

## 5. Speed and qualitative results

In [ ]:
!python -m cityseg.benchmark --checkpoint {CHECKPOINT} --size 1024 2048

In [ ]:
!python -m cityseg.visualize --checkpoint {CHECKPOINT} --data-root {DATA_ROOT} --num 4 --output {OUTPUT_DIR}/qualitative.png
from IPython.display import Image as show
show(f'{OUTPUT_DIR}/qualitative.png')

## 6. Predict on your own images

Point `INPUT` at any image or folder; results keep the same file names and resolution.

In [ ]:
INPUT = f'{DATA_ROOT}/leftImg8bit/test/munster'
!python -m cityseg.predict --checkpoint {CHECKPOINT} --input {INPUT} --output /content/results --save color overlay